# 🔥 XAUUSD Scalping ML — M1 + M5 Analysis

> **Data:** MT5 export format (`TAB` separated)  
> **Timeframe:** M1 (entry) + M5 (trend filter)  
> **Symbol:** XAUUSDm (Gold/USD micro)  
> **Strategy:** Multi-timeframe ML scalping

---
| Step | Isi |
|------|-----|
| 1 | Load & Fix MT5 Data |
| 2 | EDA + Spread Analysis |
| 3 | Multi-TF Merge (M1 ← M5 trend) |
| 4 | Feature Engineering (50+ fitur) |
| 5 | Target Creation (TP/SL dollar-based) |
| 6 | Preprocessing + Feature Selection |
| 7 | Baseline + Ensemble Models |
| 8 | Optuna Hyperparameter Tuning |
| 9 | Walk-Forward Validation |
| 10 | SHAP Feature Importance |
| 11 | Backtest dengan Spread Cost |
| 12 | Export Model |

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 0. INSTALL DEPENDENCIES
# ══════════════════════════════════════════════════════════════════
import subprocess, sys

required = {
    'lightgbm'   : 'lightgbm',
    'xgboost'    : 'xgboost',
    'sklearn'    : 'scikit-learn',
    'optuna'     : 'optuna',
    'shap'       : 'shap',
    'plotly'     : 'plotly',
    'ta'         : 'ta',
}

for imp, pkg in required.items():
    try:
        __import__(imp)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ Semua library siap!')

✅ Semua library siap!


In [ ]:
# ══════════════════════════════════════════════════════════════════
# 1. IMPORTS + CONFIG
# ══════════════════════════════════════════════════════════════════
import warnings, json
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    VotingClassifier, StackingClassifier, GradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, classification_report, confusion_matrix
)
import optuna
import shap
import joblib

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Paths
HERE = Path('.')
M1_PATH = HERE / 'XAUUSDm_M1.csv'
M5_PATH = HERE / 'XAUUSDm_M5.csv'

# ── Visual
plt.style.use('dark_background')
sns.set_palette('husl')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
SEED = 42
np.random.seed(SEED)

# ── Trading Params
TP_USD       = 2.0    # Take Profit dalam USD / ounce
SL_USD       = 1.0    # Stop Loss dalam USD / ounce
LOOKAHEAD    = 10     # Max candle M1 untuk cek TP/SL
SPREAD_LIMIT = 500     # Filter candle dengan spread > 30 (dalam poin MT5)
K_FEATURES   = 50     # Fitur terpilih untuk model

print(f'lightgbm {lgb.__version__} | xgboost {xgb.__version__}')
print(f'M1 path exists: {M1_PATH.exists()}')
print(f'M5 path exists: {M5_PATH.exists()}')

ModuleNotFoundError: No module named 'matplotlib'

---
## 📂 Step 1 — Load & Fix MT5 Data

In [ ]:
# ══════════════════════════════════════════════════════════════════
# LOAD MT5 FORMAT: TAB-separated, header pakai <>
# ══════════════════════════════════════════════════════════════════
def load_mt5(path: str) -> pd.DataFrame:
    """Load file export MT5 (tab-separated, header <DATE> <TIME> ...)"""
    df = pd.read_csv(path, sep='\t')

    # Bersihkan nama kolom dari angle brackets
    df.columns = [
        c.strip().replace('<','').replace('>','').lower()
        for c in df.columns
    ]

    # Fix format tanggal MT5: 2025.12.16 → 2025-12-16
    df['datetime'] = pd.to_datetime(
        df['date'].str.replace('.', '-', regex=False) + ' ' + df['time']
    )

    df = df.sort_values('datetime').reset_index(drop=True)
    df = df.set_index('datetime')
    df = df.drop(columns=['date','time'], errors='ignore')

    # Rename spread jadi spread_pt (spread dalam poin)
    if 'spread' in df.columns:
        df = df.rename(columns={'spread': 'spread_pt', 'tickvol': 'volume', 'vol': 'vol_real'})

    # Tipe data
    for col in ['open','high','low','close']:
        df[col] = df[col].astype(float)

    return df

# ── Load
m1 = load_mt5(M1_PATH)
m5 = load_mt5(M5_PATH)

print('=== M1 (1-Menit) ===')
print(f'  Rows    : {len(m1):,}')
print(f'  Range   : {m1.index[0]} → {m1.index[-1]}')
print(f'  Columns : {list(m1.columns)}')
print()
print('=== M5 (5-Menit) ===')
print(f'  Rows    : {len(m5):,}')
print(f'  Range   : {m5.index[0]} → {m5.index[-1]}')
print(f'  Columns : {list(m5.columns)}')
print()
print('--- M1 sample ---')
m1.head(3)

In [ ]:
# ── Statistik Dasar
print('=== M1 Statistics ===')
m1[['open','high','low','close','volume','spread_pt']].describe().round(3)

---
## 📊 Step 2 — EDA + Spread Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EDA OVERVIEW
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(3, 2, figsize=(18, 14))

# 1. Harga Close M5
axes[0,0].plot(m5['close'], lw=0.5, color='#00d4ff')
axes[0,0].set_title('XAUUSD Close Price (M5)', fontsize=12)
axes[0,0].set_ylabel('Price (USD)')

# 2. Spread Distribution M1
axes[0,1].hist(m1['spread_pt'], bins=60, color='#ff6b35', edgecolor='none', alpha=0.8)
axes[0,1].axvline(SPREAD_LIMIT, color='yellow', lw=2, ls='--', label=f'Limit={SPREAD_LIMIT}')
pct_filtered = (m1['spread_pt'] > SPREAD_LIMIT).mean() * 100
axes[0,1].set_title(f'Spread Distribution M1\n({pct_filtered:.1f}% candles akan difilter)')
axes[0,1].legend()

# 3. Volume M1 (TickVol)
vol_hourly = m1['volume'].resample('1h').sum()
axes[1,0].bar(vol_hourly.index, vol_hourly.values, color='#c3f0ca', alpha=0.7, width=0.03)
axes[1,0].set_title('Volume per Jam (M1 TickVol)')

# 4. Return Distribution M1
ret = m1['close'].pct_change().dropna() * 100
axes[1,1].hist(ret, bins=150, color='#a8edea', edgecolor='none', alpha=0.8)
axes[1,1].axvline(ret.mean(),  color='lime',   lw=1.5, label=f'Mean={ret.mean():.4f}%')
axes[1,1].axvline(ret.std(),   color='yellow', lw=1.5, ls='--', label=f'Std={ret.std():.4f}%')
axes[1,1].set_title('M1 Return Distribution')
axes[1,1].legend()
axes[1,1].set_xlim(-1, 1)

# 5. Rata-rata Spread per Jam
m1_tmp = m1.copy()
m1_tmp['hour'] = m1_tmp.index.hour
spread_hour = m1_tmp.groupby('hour')['spread_pt'].mean()
colors_bar = ['#ff4466' if v > SPREAD_LIMIT else '#00ff88' for v in spread_hour]
axes[2,0].bar(spread_hour.index, spread_hour.values, color=colors_bar)
axes[2,0].axhline(SPREAD_LIMIT, color='yellow', lw=2, ls='--', label=f'Limit={SPREAD_LIMIT}')
axes[2,0].set_title('Avg Spread per Hour (UTC)')
axes[2,0].set_xlabel('Hour (UTC)')
axes[2,0].legend()

# 6. Candle Range (High-Low) per Jam
m1_tmp['range'] = m1_tmp['high'] - m1_tmp['low']
range_hour = m1_tmp.groupby('hour')['range'].mean()
axes[2,1].bar(range_hour.index, range_hour.values, color='#f7971e')
axes[2,1].set_title('Avg Candle Range per Hour (USD)')
axes[2,1].set_xlabel('Hour (UTC)')

plt.suptitle('XAUUSD M1 + M5 — EDA Overview', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_xauusd.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n⚠️  {pct_filtered:.1f}% candle M1 punya spread > {SPREAD_LIMIT} poin (akan difilter)')

In [ ]:
# ── Sesi Trading Heatmap
def get_session(hour: int) -> str:
    if 0  <= hour < 7:  return 'Asia'
    if 7  <= hour < 12: return 'London'
    if 12 <= hour < 20: return 'New York'
    return 'Off-Hours'

m1_tmp['session'] = m1_tmp['hour'].apply(get_session)
m1_tmp['dow']     = m1_tmp.index.dayofweek
m1_tmp['ret']     = m1_tmp['close'].pct_change()

session_stats = m1_tmp.groupby('session').agg(
    avg_range    = ('range', 'mean'),
    avg_spread   = ('spread_pt', 'mean'),
    avg_volume   = ('volume', 'mean'),
    candles      = ('close', 'count')
).round(3)

print('📅 Statistik per Sesi Trading (UTC):')
print(session_stats.to_string())

print('\n💡 Rekomendasi:')
best_sess = session_stats['avg_range'].idxmax()
print(f'   Sesi terbaik untuk scalping: {best_sess} (range terbesar)')

---
## 🔗 Step 3 — Multi-Timeframe Merge (M1 ← M5)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FEATURE M5 — Trend + Indikator Higher Timeframe
# ══════════════════════════════════════════════════════════════════
def add_m5_features(m5: pd.DataFrame) -> pd.DataFrame:
    d = m5.copy()
    c = d['close']; h = d['high']; l = d['low']

    # ── Trend EMAs
    d['m5_ema20']  = c.ewm(span=20, adjust=False).mean()
    d['m5_ema50']  = c.ewm(span=50, adjust=False).mean()
    d['m5_ema200'] = c.ewm(span=200, adjust=False).mean()

    d['m5_trend_ema20']  = (c > d['m5_ema20']).astype(int)   # 1=bullish, 0=bearish
    d['m5_trend_ema50']  = (c > d['m5_ema50']).astype(int)
    d['m5_trend_ema200'] = (c > d['m5_ema200']).astype(int)

    # ── RSI
    delta = c.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    d['m5_rsi'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

    # ── MACD
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    d['m5_macd']      = ema12 - ema26
    d['m5_macd_sig']  = d['m5_macd'].ewm(span=9, adjust=False).mean()
    d['m5_macd_hist'] = d['m5_macd'] - d['m5_macd_sig']

    # ── ATR
    tr = pd.concat([
        h - l,
        (h - c.shift()).abs(),
        (l - c.shift()).abs()
    ], axis=1).max(axis=1)
    d['m5_atr'] = tr.ewm(span=14, adjust=False).mean()

    # ── ADX
    dm_pos = (h - h.shift()).clip(lower=0)
    dm_neg = (l.shift() - l).clip(lower=0)
    di_pos = 100 * dm_pos.ewm(span=14).mean() / (d['m5_atr'] + 1e-9)
    di_neg = 100 * dm_neg.ewm(span=14).mean() / (d['m5_atr'] + 1e-9)
    dx = 100 * (di_pos - di_neg).abs() / (di_pos + di_neg + 1e-9)
    d['m5_adx'] = dx.ewm(span=14).mean()

    # ── Bollinger Bands
    bb_mid  = c.rolling(20).mean()
    bb_std  = c.rolling(20).std()
    d['m5_bb_upper'] = bb_mid + 2 * bb_std
    d['m5_bb_lower'] = bb_mid - 2 * bb_std
    d['m5_bb_pct']   = (c - d['m5_bb_lower']) / (d['m5_bb_upper'] - d['m5_bb_lower'] + 1e-9)
    d['m5_bb_width'] = (d['m5_bb_upper'] - d['m5_bb_lower']) / (bb_mid + 1e-9)

    # ── Candle range M5
    d['m5_candle_range'] = h - l
    d['m5_body']         = (c - d['open']).abs()

    # ── Supertrend (simplified)
    mult = 3.0
    hl2  = (h + l) / 2
    band_up   = hl2 + mult * d['m5_atr']
    band_down = hl2 - mult * d['m5_atr']
    supertrend = pd.Series(np.nan, index=d.index)
    direction  = pd.Series(1, index=d.index)

    for i in range(1, len(d)):
        prev_c = c.iloc[i-1]
        curr_c = c.iloc[i]
        bu = band_up.iloc[i];  bd = band_down.iloc[i]
        pbu = band_up.iloc[i-1]; pbd = band_down.iloc[i-1]

        final_bu = min(bu, pbu) if prev_c <= pbu else bu
        final_bd = max(bd, pbd) if prev_c >= pbd else bd

        if direction.iloc[i-1] == 1:
            direction.iloc[i] = -1 if curr_c < final_bd else 1
        else:
            direction.iloc[i] = 1 if curr_c > final_bu else -1

    d['m5_supertrend'] = (direction == 1).astype(int)  # 1=uptrend

    m5_cols = [c for c in d.columns if c.startswith('m5_')]
    return d[m5_cols]

m5_feat = add_m5_features(m5)
print(f'M5 features: {list(m5_feat.columns)}')

In [ ]:
# ── Merge M1 + M5 (backward fill: M1 pakai nilai M5 terakhir)
m1_reset = m1.reset_index()
m5_reset = m5_feat.reset_index()

df = pd.merge_asof(
    m1_reset.sort_values('datetime'),
    m5_reset.sort_values('datetime'),
    on='datetime',
    direction='backward'
).set_index('datetime')

print(f'Merged DataFrame: {df.shape}')
print(f'M5 features merged: {[c for c in df.columns if c.startswith("m5_")]}')
df.head(3)

---
## ⚙️ Step 4 — Feature Engineering M1

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING — KHUSUS XAUUSD SCALPING
# ══════════════════════════════════════════════════════════════════
def add_m1_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    c = d['close']; h = d['high']; l = d['low']; o = d['open']
    v = d['volume']

    # ──────────────────────────────────────
    # A. CANDLE STRUCTURE
    # ──────────────────────────────────────
    d['body']        = c - o
    d['body_abs']    = d['body'].abs()
    d['upper_wick']  = h - d[['open','close']].max(axis=1)
    d['lower_wick']  = d[['open','close']].min(axis=1) - l
    d['candle_range']= h - l
    d['body_pct']    = d['body_abs'] / (d['candle_range'] + 1e-9)
    d['is_bullish']  = (c > o).astype(int)
    d['is_doji']     = (d['body_pct'] < 0.1).astype(int)

    # ──────────────────────────────────────
    # B. RETURNS (multi-period)
    # ──────────────────────────────────────
    for n in [1, 2, 3, 5, 10, 15, 20]:
        d[f'ret_{n}'] = c.pct_change(n)

    # ──────────────────────────────────────
    # C. EMAs (M1 — fast)
    # ──────────────────────────────────────
    for s in [5, 9, 14, 21, 50]:
        d[f'ema{s}'] = c.ewm(span=s, adjust=False).mean()

    d['ema5_vs_9']    = d['ema5']  - d['ema9']
    d['ema9_vs_21']   = d['ema9']  - d['ema21']
    d['price_vs_ema9']= (c - d['ema9']) / (d['ema9'] + 1e-9)
    d['price_vs_ema21']=(c - d['ema21'])/ (d['ema21']+ 1e-9)

    # ──────────────────────────────────────
    # D. RSI M1
    # ──────────────────────────────────────
    delta = c.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    d['rsi']       = 100 - (100 / (1 + gain / (loss + 1e-9)))
    d['rsi_lag1']  = d['rsi'].shift(1)
    d['rsi_delta'] = d['rsi'] - d['rsi_lag1']
    d['rsi_ob']    = (d['rsi'] > 70).astype(int)  # overbought
    d['rsi_os']    = (d['rsi'] < 30).astype(int)  # oversold

    # ──────────────────────────────────────
    # E. MACD M1
    # ──────────────────────────────────────
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    d['macd']          = ema12 - ema26
    d['macd_signal']   = d['macd'].ewm(span=9, adjust=False).mean()
    d['macd_hist']     = d['macd'] - d['macd_signal']
    d['macd_hist_lag1']= d['macd_hist'].shift(1)
    d['macd_cross_up'] = ((d['macd'] > d['macd_signal']) &
                          (d['macd'].shift(1) <= d['macd_signal'].shift(1))).astype(int)

    # ──────────────────────────────────────
    # F. ATR M1 (Volatility)
    # ──────────────────────────────────────
    tr = pd.concat([
        h - l,
        (h - c.shift()).abs(),
        (l - c.shift()).abs()
    ], axis=1).max(axis=1)
    d['atr'] = tr.ewm(span=14, adjust=False).mean()
    d['atr5']= tr.ewm(span=5, adjust=False).mean()

    # Range relatif terhadap ATR
    d['range_atr']   = d['candle_range'] / (d['atr'] + 1e-9)
    d['body_atr']    = d['body_abs']     / (d['atr'] + 1e-9)

    # ──────────────────────────────────────
    # G. BOLLINGER BANDS M1
    # ──────────────────────────────────────
    bb_mid   = c.rolling(20).mean()
    bb_std   = c.rolling(20).std()
    bb_upper = bb_mid + 2 * bb_std
    bb_lower = bb_mid - 2 * bb_std
    d['bb_pct']   = (c - bb_lower) / (bb_upper - bb_lower + 1e-9)
    d['bb_width'] = (bb_upper - bb_lower) / (bb_mid + 1e-9)
    d['bb_upper_dist'] = (bb_upper - c) / (d['atr'] + 1e-9)
    d['bb_lower_dist'] = (c - bb_lower) / (d['atr'] + 1e-9)

    # ──────────────────────────────────────
    # H. STOCHASTIC M1
    # ──────────────────────────────────────
    lo14 = l.rolling(14).min()
    hi14 = h.rolling(14).max()
    d['stoch_k'] = 100 * (c - lo14) / (hi14 - lo14 + 1e-9)
    d['stoch_d'] = d['stoch_k'].rolling(3).mean()

    # ──────────────────────────────────────
    # I. MOMENTUM & PRICE STRUCTURE
    # ──────────────────────────────────────
    d['momentum3']  = c - c.shift(3)
    d['momentum5']  = c - c.shift(5)
    d['momentum10'] = c - c.shift(10)

    # Higher High / Lower Low
    d['hh3'] = (h > h.rolling(3).max().shift(1)).astype(int)
    d['ll3'] = (l < l.rolling(3).min().shift(1)).astype(int)

    # ──────────────────────────────────────
    # J. VOLUME FEATURES
    # ──────────────────────────────────────
    d['vol_ratio5']  = v / (v.rolling(5).mean()  + 1e-9)
    d['vol_ratio20'] = v / (v.rolling(20).mean() + 1e-9)
    d['vol_delta']   = v - v.shift(1)

    # ──────────────────────────────────────
    # K. SPREAD FEATURES
    # ──────────────────────────────────────
    d['spread_norm'] = d['spread_pt'] / (d['candle_range'] * 1000 + 1e-9)  # relatif
    d['high_spread'] = (d['spread_pt'] > SPREAD_LIMIT).astype(int)

    # ──────────────────────────────────────
    # L. SESSION & TIME
    # ──────────────────────────────────────
    d['hour']        = d.index.hour
    d['minute']      = d.index.minute
    d['dow']         = d.index.dayofweek
    d['is_london']   = ((d['hour'] >= 7)  & (d['hour'] < 12)).astype(int)
    d['is_ny']       = ((d['hour'] >= 12) & (d['hour'] < 20)).astype(int)
    d['is_asia']     = ((d['hour'] >= 0)  & (d['hour'] < 7)).astype(int)
    d['is_overlap']  = ((d['hour'] >= 12) & (d['hour'] < 16)).astype(int)  # London+NY

    # ──────────────────────────────────────
    # M. CANDLE PATTERNS
    # ──────────────────────────────────────
    # Hammer: lower wick >> body, kecil upper wick
    d['hammer'] = (
        (d['lower_wick'] > 2 * d['body_abs']) &
        (d['upper_wick'] < d['body_abs']) &
        (d['body_pct'] < 0.4)
    ).astype(int)

    # Shooting star
    d['shooting_star'] = (
        (d['upper_wick'] > 2 * d['body_abs']) &
        (d['lower_wick'] < d['body_abs']) &
        (d['body_pct'] < 0.4)
    ).astype(int)

    # Engulfing bullish
    d['bull_engulf'] = (
        (c > o) &
        (c.shift(1) < o.shift(1)) &
        (c > o.shift(1)) &
        (o < c.shift(1))
    ).astype(int)

    # Engulfing bearish
    d['bear_engulf'] = (
        (c < o) &
        (c.shift(1) > o.shift(1)) &
        (c < o.shift(1)) &
        (o > c.shift(1))
    ).astype(int)

    # ──────────────────────────────────────
    # N. TREND CONFLICT (M1 vs M5)
    # ──────────────────────────────────────
    m1_up = (c > d['ema21']).astype(int)
    if 'm5_trend_ema50' in d.columns:
        d['trend_align'] = (m1_up == d['m5_trend_ema50']).astype(int)

    return d

df = add_m1_features(df)
print(f'Total kolom setelah feature engineering: {len(df.columns)}')
print(f'Sample features: {[c for c in df.columns if not c.startswith("m5_")][:20]}')

---
## 🎯 Step 5 — Target Creation (TP/SL Dollar-Based)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TARGET CREATION — KHUSUS XAUUSD
# TP = 2.0 USD, SL = 1.0 USD  → RR = 1:2
# ══════════════════════════════════════════════════════════════════
print(f'TP = {TP_USD} USD | SL = {SL_USD} USD | Lookahead = {LOOKAHEAD} candle M1')
print('Membuat label... (mungkin 1-2 menit untuk 100k baris)')

def create_target_vectorized(df: pd.DataFrame,
                              tp: float = TP_USD,
                              sl: float = SL_USD,
                              lookahead: int = LOOKAHEAD) -> pd.Series:
    """
    Vectorized target creation:
    1 = BUY signal → price akan naik TP sebelum kena SL
    0 = SELL/WAIT  → otherwise
    """
    close_arr = df['close'].values
    high_arr  = df['high'].values
    low_arr   = df['low'].values
    n         = len(df)
    labels    = np.full(n, np.nan)

    for i in range(n - lookahead):
        entry  = close_arr[i]
        tp_lvl = entry + tp
        sl_lvl = entry - sl

        hit_tp = False
        hit_sl = False

        for j in range(i+1, min(i+1+lookahead, n)):
            if high_arr[j] >= tp_lvl:
                hit_tp = True
                break
            if low_arr[j] <= sl_lvl:
                hit_sl = True
                break

        if hit_tp and not hit_sl:
            labels[i] = 1   # BUY WIN
        else:
            labels[i] = 0   # MISS

    return pd.Series(labels, index=df.index)

df['target'] = create_target_vectorized(df, TP_USD, SL_USD, LOOKAHEAD)

# Statistik label
n_valid = df['target'].notna().sum()
n_buy   = (df['target'] == 1).sum()
n_miss  = (df['target'] == 0).sum()

print(f'\n✅ Label selesai!')
print(f'   Valid labels : {n_valid:,}')
print(f'   BUY (1)      : {n_buy:,} ({n_buy/n_valid*100:.1f}%)')
print(f'   MISS (0)     : {n_miss:,} ({n_miss/n_valid*100:.1f}%)')

# Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(['BUY (1)','MISS (0)'], [n_buy, n_miss],
            color=['#00ff88','#ff4466'], edgecolor='none')
axes[0].set_title(f'Target Distribution (TP={TP_USD}$ SL={SL_USD}$)')
for ax_bar, val in zip(axes[0].patches, [n_buy, n_miss]):
    axes[0].text(ax_bar.get_x() + ax_bar.get_width()/2,
                 ax_bar.get_height() + 50, f'{val:,}', ha='center')

# Rolling win rate
win_roll = df['target'].rolling(500).mean()
axes[1].plot(win_roll, lw=0.8, color='#00d4ff')
axes[1].axhline(0.5, color='yellow', lw=1, ls='--', label='50%')
axes[1].set_title('Rolling Win Rate (500 candles)')
axes[1].set_ylabel('Win Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔧 Step 6 — Preprocessing + Feature Selection

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PREPROCESSING
# ══════════════════════════════════════════════════════════════════

# ── Filter 1: Hapus baris dengan spread tinggi (news spike)
df_clean = df[
    df['target'].notna() &
    (df['spread_pt'] <= SPREAD_LIMIT)
].copy()
print(f'Setelah spread filter (≤{SPREAD_LIMIT}): {len(df_clean):,} baris')

# ── Pilih feature columns
EXCLUDE = [
    'target','open','high','low','close','volume','vol_real',
    'spread_pt','spread_norm','high_spread'
]

feat_cols = [
    c for c in df_clean.columns
    if c not in EXCLUDE
    and df_clean[c].dtype in [np.float64, np.float32, np.int64, np.int32, int, float]
]

X_all = df_clean[feat_cols].copy()
y_all = df_clean['target'].astype(int)

# ── Hapus kolom NaN > 20%
null_pct = X_all.isnull().mean()
bad_cols  = null_pct[null_pct > 0.2].index.tolist()
X_all.drop(columns=bad_cols, inplace=True)
print(f'Dropped {len(bad_cols)} cols dengan >20% NaN')

# ── Fill NaN + Inf
X_all = X_all.ffill().fillna(0)
X_all = X_all.replace([np.inf, -np.inf], 0)

print(f'\nFeature matrix  : {X_all.shape}')
print(f'Label balance   : BUY={y_all.mean()*100:.1f}%')

In [ ]:
# ── Train/Test Split (Chronological 80/20)
N       = len(X_all)
SPLIT   = int(N * 0.80)

X_train = X_all.iloc[:SPLIT]
X_test  = X_all.iloc[SPLIT:]
y_train = y_all.iloc[:SPLIT]
y_test  = y_all.iloc[SPLIT:]

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Train date range: {X_train.index[0]} → {X_train.index[-1]}')
print(f'Test  date range: {X_test.index[0]} → {X_test.index[-1]}')

# ── Scaling (RobustScaler tahan outlier)
scaler      = RobustScaler()
X_train_s   = scaler.fit_transform(X_train)
X_test_s    = scaler.transform(X_test)

# ── Feature Selection (Mutual Info)
selector    = SelectKBest(mutual_info_classif, k=K_FEATURES)
X_train_sel = selector.fit_transform(X_train_s, y_train)
X_test_sel  = selector.transform(X_test_s)

sel_features = np.array(X_train.columns)[selector.get_support()].tolist()
mi_scores    = pd.Series(
    selector.scores_[selector.get_support()],
    index=sel_features
).sort_values(ascending=False)

print(f'\nTop 15 features (Mutual Info):')
print(mi_scores.head(15).to_string())

In [ ]:
# ── Feature Importance Chart
fig, ax = plt.subplots(figsize=(10, 10))
top25 = mi_scores.head(25)
ax.barh(top25.index[::-1], top25.values[::-1], color='#00d4ff')
ax.set_title('Top 25 Features — Mutual Information Score')
ax.set_xlabel('MI Score')
plt.tight_layout()
plt.savefig('mutual_info.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🤖 Step 7 — Baseline + Ensemble Models

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EVALUASI HELPER
# ══════════════════════════════════════════════════════════════════
RESULTS = {}

def eval_model(name, model, Xtr, ytr, Xte, yte, conf_thresh=0.65):
    model.fit(Xtr, ytr)

    prob = model.predict_proba(Xte)[:, 1]
    pred = (prob >= 0.5).astype(int)

    acc   = accuracy_score(yte, pred)
    prec  = precision_score(yte, pred, zero_division=0)
    rec   = recall_score(yte, pred, zero_division=0)
    f1    = f1_score(yte, pred, zero_division=0)
    auc   = roc_auc_score(yte, prob)

    # Confident prediction (prob >= conf_thresh atau <= 1-conf_thresh)
    conf_mask = (prob >= conf_thresh) | (prob <= (1 - conf_thresh))
    conf_acc  = accuracy_score(yte[conf_mask], pred[conf_mask]) if conf_mask.sum() > 5 else 0
    conf_cov  = conf_mask.mean() * 100

    RESULTS[name] = dict(
        acc=acc, prec=prec, rec=rec, f1=f1,
        auc=auc, conf_acc=conf_acc, conf_cov=conf_cov
    )

    bar = '█' * int(acc * 30)
    print(f'[{name:<28}] Acc={acc*100:5.1f}%  Prec={prec*100:5.1f}%  '
          f'F1={f1*100:5.1f}%  AUC={auc:.3f}  ConfAcc={conf_acc*100:5.1f}% ({conf_cov:.0f}%)')
    return model

print('=== BASELINE ===')
lr_m = eval_model('LogReg',
    LogisticRegression(max_iter=1000, C=0.1, random_state=SEED),
    X_train_sel, y_train, X_test_sel, y_test
)

print('\n=== TREE MODELS ===')
rf_m = eval_model('RandomForest',
    RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=10,
                           n_jobs=-1, random_state=SEED),
    X_train_sel, y_train, X_test_sel, y_test
)

et_m = eval_model('ExtraTrees',
    ExtraTreesClassifier(n_estimators=200, max_depth=8, min_samples_leaf=10,
                         n_jobs=-1, random_state=SEED),
    X_train_sel, y_train, X_test_sel, y_test
)

print('\n=== GRADIENT BOOSTING ===')
lgb_m = eval_model('LightGBM',
    lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                       num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                       min_child_samples=30, random_state=SEED, verbose=-1),
    X_train_sel, y_train, X_test_sel, y_test
)

xgb_m = eval_model('XGBoost',
    xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=5,
                      subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
                      eval_metric='logloss', use_label_encoder=False,
                      random_state=SEED, verbosity=0),
    X_train_sel, y_train, X_test_sel, y_test
)

In [ ]:
# ── Voting + Stacking
print('=== ENSEMBLE ===')

voting_m = eval_model('Voting(LGB+XGB+RF)',
    VotingClassifier(
        estimators=[
            ('lgb', lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                                        num_leaves=31, random_state=SEED, verbose=-1)),
            ('xgb', xgb.XGBClassifier(n_estimators=500, learning_rate=0.05,
                                       eval_metric='logloss', use_label_encoder=False,
                                       random_state=SEED, verbosity=0)),
            ('rf',  RandomForestClassifier(n_estimators=200, max_depth=8,
                                            min_samples_leaf=10, n_jobs=-1, random_state=SEED))
        ],
        voting='soft', weights=[2, 2, 1]
    ),
    X_train_sel, y_train, X_test_sel, y_test
)

stacking_m = eval_model('Stacking(LGB+XGB+ET→LR)',
    StackingClassifier(
        estimators=[
            ('lgb', lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                        random_state=SEED, verbose=-1)),
            ('xgb', xgb.XGBClassifier(n_estimators=300, learning_rate=0.05,
                                       eval_metric='logloss', use_label_encoder=False,
                                       random_state=SEED, verbosity=0)),
            ('et',  ExtraTreesClassifier(n_estimators=200, max_depth=7,
                                          n_jobs=-1, random_state=SEED))
        ],
        final_estimator=CalibratedClassifierCV(
            LogisticRegression(C=1.0, max_iter=500), method='isotonic'
        ),
        cv=3, stack_method='predict_proba', n_jobs=-1
    ),
    X_train_sel, y_train, X_test_sel, y_test
)

---
## 🎯 Step 8 — Optuna Hyperparameter Tuning

In [ ]:
# ══════════════════════════════════════════════════════════════════
# OPTUNA — LightGBM
# ══════════════════════════════════════════════════════════════════
N_TRIALS = 15   # Naikkan ke 100+ untuk akurasi lebih tinggi
tscv5    = TimeSeriesSplit(n_splits=5)

def lgb_objective(trial):
    p = {
        'n_estimators'       : trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate'      : trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'max_depth'          : trial.suggest_int('max_depth', 3, 10),
        'num_leaves'         : trial.suggest_int('num_leaves', 15, 100),
        'subsample'          : trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree'   : trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_samples'  : trial.suggest_int('min_child_samples', 10, 150),
        'reg_alpha'          : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'         : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'class_weight'       : 'balanced',
        'random_state'       : SEED,
        'verbose'            : -1,
        'n_jobs'             : -1
    }
    m   = lgb.LGBMClassifier(**p)
    sc  = cross_val_score(m, X_train_sel, y_train, cv=tscv5, scoring='f1', n_jobs=-1)
    return sc.mean()

study_lgb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_lgb.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ LGB Best F1: {study_lgb.best_value:.4f}')
print(f'   Best params: {study_lgb.best_params}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# OPTUNA — XGBoost
# ══════════════════════════════════════════════════════════════════
def xgb_objective(trial):
    p = {
        'n_estimators'    : trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'max_depth'       : trial.suggest_int('max_depth', 3, 10),
        'subsample'       : trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 50),
        'gamma'           : trial.suggest_float('gamma', 0, 10),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'scale_pos_weight': y_train.value_counts()[0] / y_train.value_counts()[1],
        'eval_metric'     : 'logloss',
        'use_label_encoder': False,
        'random_state'    : SEED,
        'verbosity'       : 0,
        'n_jobs'          : -1
    }
    m  = xgb.XGBClassifier(**p)
    sc = cross_val_score(m, X_train_sel, y_train, cv=tscv5, scoring='f1', n_jobs=-1)
    return sc.mean()

study_xgb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_xgb.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ XGB Best F1: {study_xgb.best_value:.4f}')
print(f'   Best params: {study_xgb.best_params}')

In [ ]:
# ── Train & Evaluate Tuned Models
print('=== TUNED MODELS ===')

lgb_tuned = eval_model('LGB_Tuned',
    lgb.LGBMClassifier(**study_lgb.best_params),
    X_train_sel, y_train, X_test_sel, y_test
)

xgb_tuned = eval_model('XGB_Tuned',
    xgb.XGBClassifier(**study_xgb.best_params),
    X_train_sel, y_train, X_test_sel, y_test
)

# Tuned Voting
voting_tuned = eval_model('Voting_Tuned(LGB+XGB+RF)',
    VotingClassifier(
        estimators=[
            ('lgb', lgb.LGBMClassifier(**study_lgb.best_params)),
            ('xgb', xgb.XGBClassifier(**study_xgb.best_params)),
            ('rf',  RandomForestClassifier(n_estimators=300, max_depth=8,
                                            min_samples_leaf=10, n_jobs=-1, random_state=SEED))
        ],
        voting='soft', weights=[2, 2, 1]
    ),
    X_train_sel, y_train, X_test_sel, y_test
)

---
## ⏱️ Step 9 — Walk-Forward Validation

In [ ]:
# ══════════════════════════════════════════════════════════════════
# WALK-FORWARD — 5 FOLD TIME SERIES CV
# ══════════════════════════════════════════════════════════════════
WF_SPLITS = 5
tscv_wf   = TimeSeriesSplit(n_splits=WF_SPLITS)

def run_walkforward(name, ModelClass, params, X, y):
    fold_stats = []

    for fold, (tr_idx, te_idx) in enumerate(tscv_wf.split(X)):
        m = ModelClass(**params)
        m.fit(X[tr_idx], y.iloc[tr_idx])
        prob = m.predict_proba(X[te_idx])[:, 1]
        pred = (prob >= 0.5).astype(int)

        acc  = accuracy_score(y.iloc[te_idx], pred)
        f1   = f1_score(y.iloc[te_idx], pred, zero_division=0)
        prec = precision_score(y.iloc[te_idx], pred, zero_division=0)

        conf_mask = (prob >= 0.65) | (prob <= 0.35)
        conf_acc  = accuracy_score(y.iloc[te_idx][conf_mask], pred[conf_mask]) \
                    if conf_mask.sum() > 5 else 0

        fold_stats.append({'fold': fold+1, 'acc': acc, 'f1': f1,
                           'prec': prec, 'conf_acc': conf_acc,
                           'n_samples': len(te_idx)})
        print(f'  [{name}] Fold {fold+1}: Acc={acc*100:.1f}%  F1={f1*100:.1f}%  '
              f'Prec={prec*100:.1f}%  ConfAcc={conf_acc*100:.1f}%  n={len(te_idx):,}')

    df_folds = pd.DataFrame(fold_stats)
    print(f'  ─── [{name}] Mean: Acc={df_folds.acc.mean()*100:.1f}%  '
          f'F1={df_folds.f1.mean()*100:.1f}%  ConfAcc={df_folds.conf_acc.mean()*100:.1f}%\n')
    return df_folds

lgb_params = {**study_lgb.best_params, 'n_jobs': -1}
xgb_params = {**study_xgb.best_params, 'n_jobs': -1, 'verbosity': 0}

wf_lgb = run_walkforward('LGB_Tuned', lgb.LGBMClassifier, lgb_params, X_train_sel, y_train)
wf_xgb = run_walkforward('XGB_Tuned', xgb.XGBClassifier,  xgb_params, X_train_sel, y_train)

In [ ]:
# ── Walk-Forward Chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
folds = range(1, WF_SPLITS+1)

for ax, col, title in zip(
    axes,
    ['acc','f1','conf_acc'],
    ['Accuracy','F1 Score','Confident Accuracy (≥65%)']
):
    ax.plot(folds, wf_lgb[col], 'o-', color='#00d4ff', lw=2, label='LGB Tuned', ms=8)
    ax.plot(folds, wf_xgb[col], 's-', color='#ff6b35', lw=2, label='XGB Tuned', ms=8)
    ax.fill_between(folds, wf_lgb[col], wf_xgb[col], alpha=0.1, color='white')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Fold')
    ax.legend(fontsize=9)
    ax.set_ylim(0.3, 1.0)
    ax.axhline(0.6, color='yellow', lw=1, ls='--', alpha=0.7, label='0.6 baseline')
    ax.grid(True, alpha=0.2)

plt.suptitle('Walk-Forward Validation (5-Fold TimeSeriesCV)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('walkforward.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔍 Step 10 — SHAP Feature Importance

In [ ]:
# ══════════════════════════════════════════════════════════════════
# SHAP ANALYSIS
# ══════════════════════════════════════════════════════════════════
lgb_final = lgb.LGBMClassifier(**study_lgb.best_params)
lgb_final.fit(X_train_sel, y_train)

explainer = shap.TreeExplainer(lgb_final)

# Ambil sample test untuk SHAP (max 3000 untuk speed)
n_shap   = min(3000, len(X_test_sel))
X_sample = X_test_sel[:n_shap]

shap_vals = explainer.shap_values(X_sample)
sv = shap_vals[1] if isinstance(shap_vals, list) else shap_vals

mean_shap = np.abs(sv).mean(axis=0)
shap_df   = pd.DataFrame({
    'feature'   : sel_features,
    'mean_shap' : mean_shap
}).sort_values('mean_shap', ascending=False).reset_index(drop=True)

print('Top 20 Features (SHAP):')
print(shap_df.head(20).to_string(index=False))

In [ ]:
# ── SHAP Charts
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Bar chart
top25 = shap_df.head(25)
axes[0].barh(top25['feature'][::-1], top25['mean_shap'][::-1], color='#00d4ff')
axes[0].set_title('Top 25 Features — Mean |SHAP|', fontsize=12)
axes[0].set_xlabel('Mean |SHAP value|')

# Color by feature group
colors = []
for feat in top25['feature']:
    if feat.startswith('m5_'):         colors.append('#ff6b35')
    elif 'rsi' in feat:                colors.append('#c3f0ca')
    elif 'macd' in feat:               colors.append('#f7971e')
    elif 'ema' in feat:                colors.append('#a8edea')
    elif 'ret' in feat:                colors.append('#ffd89b')
    elif 'session' in feat or 'hour' in feat: colors.append('#ff9ff3')
    else:                              colors.append('#00d4ff')
for patch, color in zip(axes[0].patches, colors):
    patch.set_facecolor(color)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#ff6b35', label='M5 features'),
    Patch(facecolor='#c3f0ca', label='RSI'),
    Patch(facecolor='#f7971e', label='MACD'),
    Patch(facecolor='#a8edea', label='EMA'),
    Patch(facecolor='#ffd89b', label='Returns'),
    Patch(facecolor='#00d4ff', label='Other'),
]
axes[0].legend(handles=legend_elements, loc='lower right', fontsize=9)

# SHAP dot plot (scatter)
top10_feats = shap_df.head(10)['feature'].tolist()
top10_idx   = [sel_features.index(f) for f in top10_feats]

for i, (idx, feat) in enumerate(zip(top10_idx, top10_feats)):
    sv_col   = sv[:, idx]
    feat_val = X_sample[:, idx]
    scatter = axes[1].scatter(
        sv_col, np.full_like(sv_col, i),
        c=feat_val, cmap='RdYlGn', alpha=0.3, s=8
    )

axes[1].set_yticks(range(len(top10_feats)))
axes[1].set_yticklabels(top10_feats, fontsize=9)
axes[1].axvline(0, color='white', lw=1)
axes[1].set_title('SHAP Scatter — Top 10 Features\n(green=high value, red=low value)', fontsize=11)
axes[1].set_xlabel('SHAP value (→ BUY, ← SELL)')

plt.tight_layout()
plt.savefig('shap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 💰 Step 11 — Backtest dengan Spread Cost

In [ ]:
# ══════════════════════════════════════════════════════════════════
# BACKTEST — REALISTIC (termasuk spread cost)
# ══════════════════════════════════════════════════════════════════
INITIAL_CAPITAL = 1_000    # USD
RISK_PER_TRADE  = 0.01     # 1% dari kapital
PROB_THRESHOLD  = 0.62     # Masuk trade jika prob >= ini
SPREAD_COST_USD = 0.20     # Estimasi spread cost per trade (USD)

# Dapatkan probabilitas model terbaik pada test set
voting_tuned.fit(X_train_sel, y_train)
prob_test = voting_tuned.predict_proba(X_test_sel)[:, 1]

# Match dengan test data OHLCV
test_data  = df_clean.iloc[SPLIT:].copy()
min_len    = min(len(prob_test), len(test_data))
test_data  = test_data.iloc[:min_len].copy()
prob_test  = prob_test[:min_len]

test_data['prob']   = prob_test
test_data['signal'] = (prob_test >= PROB_THRESHOLD).astype(int)

n_signals = test_data['signal'].sum()
print(f'Test period : {test_data.index[0]} → {test_data.index[-1]}')
print(f'Total M1 candles   : {len(test_data):,}')
print(f'Signals generated  : {n_signals:,} ({n_signals/len(test_data)*100:.1f}% dari candles)')
print(f'Probability threshold: {PROB_THRESHOLD}')

In [ ]:
# ── Backtest Engine
def run_backtest(test_df: pd.DataFrame,
                 tp: float   = TP_USD,
                 sl: float   = SL_USD,
                 lookahead: int = LOOKAHEAD,
                 init_capital: float = INITIAL_CAPITAL,
                 risk_pct: float = RISK_PER_TRADE,
                 spread_cost: float = SPREAD_COST_USD) -> tuple:

    capital  = init_capital
    peak     = capital
    max_dd   = 0.0
    trades   = []

    close_arr = test_df['close'].values
    high_arr  = test_df['high'].values
    low_arr   = test_df['low'].values
    sig_arr   = test_df['signal'].values
    prob_arr  = test_df['prob'].values
    idx       = test_df.index

    i = 0
    while i < len(test_df) - lookahead:
        if sig_arr[i] != 1:
            i += 1
            continue

        entry   = close_arr[i]
        tp_lvl  = entry + tp
        sl_lvl  = entry - sl
        risk_usd = capital * risk_pct

        result    = 'OPEN'
        exit_p    = entry
        exit_i    = i + 1

        for j in range(i+1, min(i+1+lookahead, len(test_df))):
            if high_arr[j] >= tp_lvl:
                result = 'WIN';  exit_p = tp_lvl; exit_i = j; break
            if low_arr[j] <= sl_lvl:
                result = 'LOSS'; exit_p = sl_lvl; exit_i = j; break

        if result == 'OPEN':
            exit_p = close_arr[min(i+lookahead, len(test_df)-1)]
            result = 'WIN' if exit_p > entry else 'LOSS'

        # PnL berdasarkan risk
        pnl_pts  = exit_p - entry if result == 'WIN' else -(entry - exit_p)
        pnl_usd  = risk_usd * (pnl_pts / sl) - spread_cost

        capital += pnl_usd
        peak     = max(peak, capital)
        dd       = (peak - capital) / peak * 100
        max_dd   = max(max_dd, dd)

        trades.append({
            'open_time'  : idx[i],
            'close_time' : idx[exit_i],
            'entry'      : round(entry, 3),
            'exit'       : round(exit_p, 3),
            'sl'         : round(sl_lvl, 3),
            'tp'         : round(tp_lvl, 3),
            'result'     : result,
            'pnl_usd'    : round(pnl_usd, 2),
            'prob'       : round(float(prob_arr[i]), 4),
            'capital'    : round(capital, 2),
            'drawdown'   : round(dd, 2)
        })

        i = exit_i + 1  # skip ke candle setelah exit

    trades_df = pd.DataFrame(trades)
    if len(trades_df) == 0:
        return {'total_trades': 0}, trades_df

    wins    = (trades_df['result'] == 'WIN').sum()
    total   = len(trades_df)
    sum_win = trades_df.loc[trades_df['result']=='WIN',  'pnl_usd'].sum()
    sum_los = trades_df.loc[trades_df['result']=='LOSS', 'pnl_usd'].abs().sum()

    stats = {
        'total_trades'   : total,
        'wins'           : int(wins),
        'losses'         : int(total - wins),
        'win_rate_pct'   : round(wins / total * 100, 2),
        'total_pnl_usd'  : round(trades_df['pnl_usd'].sum(), 2),
        'profit_factor'  : round(sum_win / (sum_los + 1e-9), 3),
        'max_drawdown_pct': round(max_dd, 2),
        'final_capital'  : round(capital, 2),
        'return_pct'     : round((capital - init_capital) / init_capital * 100, 2),
        'avg_win_usd'    : round(trades_df.loc[trades_df['result']=='WIN', 'pnl_usd'].mean(), 2),
        'avg_loss_usd'   : round(trades_df.loc[trades_df['result']=='LOSS','pnl_usd'].mean(), 2)
    }
    return stats, trades_df

bt_stats, bt_trades = run_backtest(test_data)

print('\n💰 BACKTEST RESULTS:')
print('─' * 45)
for k, v in bt_stats.items():
    icon = '✅' if k in ['win_rate_pct','profit_factor','return_pct'] and float(str(v)) > 0 else '  '
    print(f'  {icon} {k:<22}: {v}')

In [ ]:
# ── Equity Curve (Interactive Plotly)
if len(bt_trades) > 5:
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        row_heights=[0.55, 0.25, 0.20],
        subplot_titles=[
            f'Equity Curve | Return={bt_stats["return_pct"]}% | WinRate={bt_stats["win_rate_pct"]}%',
            'Drawdown (%)',
            'Prob Distribution per Trade'
        ],
        vertical_spacing=0.06
    )

    # Equity
    fig.add_trace(
        go.Scatter(
            x=bt_trades['close_time'], y=bt_trades['capital'],
            mode='lines', name='Equity', line=dict(color='#00d4ff', width=2)
        ), row=1, col=1
    )
    fig.add_hline(y=INITIAL_CAPITAL, line_dash='dash', line_color='gray', row=1, col=1)

    # Trades overlay
    wins_df  = bt_trades[bt_trades['result'] == 'WIN']
    loss_df  = bt_trades[bt_trades['result'] == 'LOSS']
    fig.add_trace(
        go.Scatter(x=wins_df['close_time'], y=wins_df['capital'],
                   mode='markers', name='WIN',
                   marker=dict(color='#00ff88', size=5, symbol='triangle-up')),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=loss_df['close_time'], y=loss_df['capital'],
                   mode='markers', name='LOSS',
                   marker=dict(color='#ff4466', size=5, symbol='triangle-down')),
        row=1, col=1
    )

    # Drawdown
    fig.add_trace(
        go.Scatter(x=bt_trades['close_time'], y=-bt_trades['drawdown'],
                   mode='lines', fill='tozeroy', name='Drawdown',
                   line=dict(color='#ff4466', width=1)),
        row=2, col=1
    )

    # Probability
    fig.add_trace(
        go.Scatter(x=bt_trades['close_time'], y=bt_trades['prob'],
                   mode='markers', name='Signal Prob',
                   marker=dict(
                       color=bt_trades['result'].map({'WIN': '#00ff88', 'LOSS': '#ff4466'}),
                       size=4, opacity=0.7
                   )),
        row=3, col=1
    )
    fig.add_hline(y=PROB_THRESHOLD, line_dash='dash', line_color='yellow', row=3, col=1)

    fig.update_layout(
        template='plotly_dark',
        height=750,
        title=f'XAUUSD M1 Scalping — Backtest Results<br>'
              f'Trades={bt_stats["total_trades"]} | PF={bt_stats["profit_factor"]} | '
              f'MaxDD={bt_stats["max_drawdown_pct"]}% | Capital: ${INITIAL_CAPITAL}→${bt_stats["final_capital"]}',
        showlegend=True
    )
    fig.write_html('equity_curve.html')
    fig.show()
    print('✅ equity_curve.html saved')

In [ ]:
# ── Trade Analysis Charts
if len(bt_trades) > 5:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # 1. PnL Distribution
    colors = ['#00ff88' if v > 0 else '#ff4466' for v in bt_trades['pnl_usd']]
    axes[0,0].bar(range(len(bt_trades)), bt_trades['pnl_usd'], color=colors, width=1)
    axes[0,0].axhline(0, color='white', lw=1)
    axes[0,0].set_title('PnL per Trade (USD)')

    # 2. PnL Histogram
    axes[0,1].hist(bt_trades.loc[bt_trades['result']=='WIN', 'pnl_usd'],
                   bins=30, color='#00ff88', alpha=0.7, label='WIN')
    axes[0,1].hist(bt_trades.loc[bt_trades['result']=='LOSS','pnl_usd'],
                   bins=30, color='#ff4466', alpha=0.7, label='LOSS')
    axes[0,1].set_title('PnL Distribution')
    axes[0,1].legend()

    # 3. Win Rate per Hour
    bt_trades['hour'] = pd.to_datetime(bt_trades['open_time']).dt.hour
    wr_hour = bt_trades.groupby('hour')['result'].apply(
        lambda x: (x=='WIN').mean() * 100
    )
    bar_c = ['#00ff88' if v >= 55 else '#ff4466' for v in wr_hour.values]
    axes[0,2].bar(wr_hour.index, wr_hour.values, color=bar_c)
    axes[0,2].axhline(50, color='yellow', lw=1, ls='--')
    axes[0,2].set_title('Win Rate per Hour (UTC)')
    axes[0,2].set_ylabel('%')

    # 4. Cumulative PnL
    axes[1,0].plot(bt_trades['capital'], color='#00d4ff', lw=1.5)
    axes[1,0].axhline(INITIAL_CAPITAL, color='gray', lw=1, ls='--')
    axes[1,0].set_title('Cumulative Capital (USD)')
    axes[1,0].fill_between(range(len(bt_trades)),
                            bt_trades['capital'], INITIAL_CAPITAL,
                            where=(bt_trades['capital'] >= INITIAL_CAPITAL),
                            alpha=0.2, color='#00ff88')
    axes[1,0].fill_between(range(len(bt_trades)),
                            bt_trades['capital'], INITIAL_CAPITAL,
                            where=(bt_trades['capital'] < INITIAL_CAPITAL),
                            alpha=0.2, color='#ff4466')

    # 5. Probability vs Result
    for res, color, label in [('WIN','#00ff88','WIN'), ('LOSS','#ff4466','LOSS')]:
        subset = bt_trades[bt_trades['result'] == res]
        axes[1,1].hist(subset['prob'], bins=20, color=color, alpha=0.7, label=label)
    axes[1,1].axvline(PROB_THRESHOLD, color='yellow', lw=2, ls='--',
                       label=f'Threshold={PROB_THRESHOLD}')
    axes[1,1].set_title('Signal Probability — WIN vs LOSS')
    axes[1,1].legend()

    # 6. Monthly PnL
    bt_trades['month'] = pd.to_datetime(bt_trades['close_time']).dt.to_period('M')
    monthly = bt_trades.groupby('month')['pnl_usd'].sum()
    bar_mc = ['#00ff88' if v > 0 else '#ff4466' for v in monthly.values]
    axes[1,2].bar(monthly.index.astype(str), monthly.values, color=bar_mc)
    axes[1,2].axhline(0, color='white', lw=1)
    axes[1,2].set_title('Monthly PnL (USD)')
    axes[1,2].tick_params(axis='x', rotation=30)

    plt.suptitle(f'XAUUSD M1 Scalping — Trade Analysis\n'
                 f'WinRate={bt_stats["win_rate_pct"]}%  PF={bt_stats["profit_factor"]}  '
                 f'Return={bt_stats["return_pct"]}%  MaxDD={bt_stats["max_drawdown_pct"]}%',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('trade_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Model Comparison + ROC Curves
res_df = pd.DataFrame(RESULTS).T.round(4).sort_values('conf_acc', ascending=False)
print('\n📊 Semua Model (sorted by ConfAcc):')
print(res_df.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Bar chart
metrics = ['acc','prec','f1','auc','conf_acc']
x = np.arange(len(res_df))
w = 0.18
palette = ['#00d4ff','#ff6b35','#c3f0ca','#f7971e','#ff9ff3']

for i, (m, col) in enumerate(zip(metrics, palette)):
    axes[0].bar(x + i*w, res_df[m], width=w, label=m.upper(), color=col, alpha=0.85)

axes[0].set_xticks(x + w*2)
axes[0].set_xticklabels(res_df.index, rotation=30, ha='right', fontsize=8)
axes[0].legend(fontsize=9)
axes[0].set_title('Model Performance Comparison')
axes[0].axhline(0.6, color='white', lw=1, ls='--', alpha=0.5)
axes[0].set_ylim(0, 1.05)

# ROC
axes[1].plot([0,1],[0,1],'k--',lw=1)
for (name, color) in [
    ('LGB_Tuned','#00d4ff'), ('XGB_Tuned','#ff6b35'),
    ('Voting_Tuned(LGB+XGB+RF)','#c3f0ca'), ('Stacking(LGB+XGB+ET→LR)','#f7971e')
]:
    row = RESULTS.get(name)
    if row:
        try:
            m_obj = [lgb_tuned, xgb_tuned, voting_tuned, stacking_m][
                ['LGB_Tuned','XGB_Tuned','Voting_Tuned(LGB+XGB+RF)','Stacking(LGB+XGB+ET→LR)'].index(name)
            ]
            pp  = m_obj.predict_proba(X_test_sel)[:,1]
            fpr, tpr, _ = roc_curve(y_test, pp)
            auc_v = roc_auc_score(y_test, pp)
            axes[1].plot(fpr, tpr, color=color, lw=2, label=f'{name} ({auc_v:.3f})')
        except Exception:
            pass

axes[1].set_title('ROC Curves')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 💾 Step 12 — Export Model

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TRAIN FINAL MODEL PADA FULL DATA → EXPORT
# ══════════════════════════════════════════════════════════════════

# Refit scaler + selector pada full data
X_full_s   = scaler.fit_transform(X_all)
X_full_sel = selector.fit_transform(X_full_s, y_all)

final_model = VotingClassifier(
    estimators=[
        ('lgb', lgb.LGBMClassifier(**study_lgb.best_params)),
        ('xgb', xgb.XGBClassifier(**study_xgb.best_params)),
        ('rf',  RandomForestClassifier(n_estimators=300, max_depth=8,
                                        min_samples_leaf=10, n_jobs=-1, random_state=SEED))
    ],
    voting='soft', weights=[2, 2, 1]
)
final_model.fit(X_full_sel, y_all)
print('✅ Final model trained on full data')

# ── Save Bundle
SAVE_DIR = HERE

bundle = {
    'model'         : final_model,
    'scaler'        : scaler,
    'selector'      : selector,
    'feature_cols'  : list(X_all.columns),
    'selected_feats': sel_features,
    'tp'            : TP_USD,
    'sl'            : SL_USD,
    'spread_limit'  : SPREAD_LIMIT,
    'prob_threshold': PROB_THRESHOLD,
    'metrics'       : RESULTS
}
joblib.dump(bundle, SAVE_DIR / 'xauusd_scalping_model.joblib', compress=3)

# ── Metadata
best_name = res_df.index[0]
meta = {
    'symbol'         : 'XAUUSDm',
    'timeframes'     : ['M1', 'M5'],
    'train_candles_m1': len(df_clean),
    'date_range'     : f'{df_clean.index[0]} → {df_clean.index[-1]}',
    'tp_usd'         : TP_USD,
    'sl_usd'         : SL_USD,
    'lookahead'      : LOOKAHEAD,
    'spread_filter'  : SPREAD_LIMIT,
    'n_features'     : K_FEATURES,
    'best_model'     : best_name,
    'accuracy'       : float(RESULTS.get(best_name, {}).get('acc', 0)),
    'conf_accuracy'  : float(RESULTS.get(best_name, {}).get('conf_acc', 0)),
    'f1_score'       : float(RESULTS.get(best_name, {}).get('f1', 0)),
    'auc'            : float(RESULTS.get(best_name, {}).get('auc', 0)),
    'backtest'       : bt_stats,
    'lgb_params'     : study_lgb.best_params,
    'xgb_params'     : study_xgb.best_params,
    'top10_features' : shap_df.head(10)['feature'].tolist()
}

with open(SAVE_DIR / 'xauusd_scalping_model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)

print(f'\n✅ Model  : {SAVE_DIR / "xauusd_scalping_model.joblib"}')
print(f'✅ Meta   : {SAVE_DIR / "xauusd_scalping_model_meta.json"}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 🏁 FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════
print('=' * 65)
print('  🔥 XAUUSD SCALPING ML — FINAL SUMMARY')
print('=' * 65)
print(f'  Data         : XAUUSDm M1 ({len(m1):,} candles) + M5 ({len(m5):,} candles)')
print(f'  Date range   : {df_clean.index[0]} → {df_clean.index[-1]}')
print(f'  TP/SL        : {TP_USD}$ / {SL_USD}$  (RR 1:{int(TP_USD/SL_USD)})')
print(f'  Spread filter: ≤{SPREAD_LIMIT} poin MT5')
print(f'  Features     : {K_FEATURES} dari {len(X_all.columns)} total')
print()
print('  📊 Model Rankings (Confident Accuracy):')
for i, (name, row) in enumerate(res_df.iterrows()):
    flag = '🥇' if i == 0 else ('🥈' if i == 1 else ('🥉' if i == 2 else '  '))
    print(f'  {flag} {i+1}. {name:<30}'
          f' Acc={row["acc"]*100:.1f}%'
          f' ConfAcc={row["conf_acc"]*100:.1f}%'
          f' F1={row["f1"]*100:.1f}%'
          f' AUC={row["auc"]:.3f}')
print()
print('  💰 Backtest Results:')
for k, v in bt_stats.items():
    print(f'     {k:<22}: {v}')
print()
print('  🔑 Top 5 Features (SHAP):')
for i, row in shap_df.head(5).iterrows():
    print(f'     {i+1}. {row["feature"]:<30} {row["mean_shap"]:.4f}')
print()
print('  📁 Files Generated:')
print('     xauusd_scalping_model.joblib')
print('     xauusd_scalping_model_meta.json')
print('     equity_curve.html   (interactive)')
print('     eda_xauusd.png')
print('     shap_analysis.png')
print('     trade_analysis.png')
print('=' * 65)